# 01.02 · Limpieza y troceado incremental

Crea chunks deterministas únicamente para transcripciones nuevas o modificadas y conserva la versión del troceador.

**Contrato:** `SEGURO` + cuatro daños entrenados. `SEGURO` es excluyente; los daños son multietiqueta. Los casos indeterminados se difieren y no entran al entrenamiento.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. No instala paquetes ni usa rutas personales. Revise el README de esta etapa antes de ejecutar.

In [ ]:
from pathlib import Path
import sys

def find_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('No se encontró pyproject.toml')

ROOT = find_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print('Proyecto:', ROOT)


## Configuración

In [ ]:
from moderacion_peru.incremental import TranscriptSegment, chunk_transcript
from moderacion_peru.io import append_jsonl_once, read_jsonl
SOURCE=ROOT/'datos/raw/transcripts_raw.jsonl'
OUTPUT=ROOT/'datos/processed/chunks_v2.jsonl'

## Materialización

In [ ]:
rows=[]
for video in read_jsonl(SOURCE) if SOURCE.exists() else []:
    segments=[TranscriptSegment(float(s['start']),float(s['duration']),s['text']) for s in video.get('segments',[])]
    rows.extend(chunk.to_dict() for chunk in chunk_transcript(video['video_id'],segments))
added,skipped=append_jsonl_once(OUTPUT,rows,id_field='chunk_id')
print({'generated':len(rows),'added':added,'already_present':skipped})